# StudentLife Multimodal Burnout Risk Prediction (Realistic Version)

This notebook implements a *realistic* multimodal pipeline **within a single dataset (StudentLife)**: we combine weekly stress, sleep, and activity signals per student, add temporal features, define a defensible proxy label, train models, and run **SHAP only for the final multimodal model**.


---
# SECTION 1: SETUP

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# Import libraries
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve # CORRECT IMPORT
import shap

print('✅ Libraries loaded')


In [ ]:
# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Path setup
REPO_ROOT = Path(os.getcwd()).resolve().parent
DATA_PATH = REPO_ROOT / "data" / "raw" / "studentlife"
OUTPUT_PATH = REPO_ROOT / "models"
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f'✅ Data Path: {DATA_PATH}')
print(f'✅ Output Path: {OUTPUT_PATH}')


---
# SECTION 2: DATA LOADING

In [ ]:
# Load StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")
stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")

---
# SECTION 3: DATA CLEANING

In [ ]:
# Clean StudentLife data
print("="*70)
print("🧹 CLEANING STUDENTLIFE DATA")
print("="*70)

# =========================
# STRESS DATASET
# =========================
print("\n📌 DATASET: STRESS")
print("➡ Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()
# Drop the export-artifact column named 'null' (irrelevant for modeling)
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Convert timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Clean student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)

# Rename level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

# Convert stress_level to numeric, drop NaNs, cast to float
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
stress_clean = stress_clean.dropna(subset=['stress_level'])
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)


# =========================
# ACTIVITY DATASET
# =========================
print("\n📌 DATASET: ACTIVITY")
print("➡ Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity...")

activity_clean = activity_raw.copy()
# Drop the export-artifact column named 'null' (irrelevant for modeling)
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp
activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)

# -------------------------
# IMPORTANT FIX (Activity EMA):
# The StudentLife "Activity" EMA file mixes multiple *different* question blocks.
# 'Social2' measures social interaction, while (working/other_working) and (relaxing/other_relaxing)
# measure behavioral state. These should NOT be merged into a single 'activity_level'.
#
# We therefore create 3 separate, interpretable features:
# 1) workload_score   = working + other_working
# 2) recovery_score   = relaxing + other_relaxing
# 3) social_score     = Social2
#
# This makes the feature scientifically meaningful and prevents invalid aggregation artifacts.
# -------------------------

# Ensure key columns exist (some StudentLife exports omit some columns)
for col in ["Social2","working","other_working","relaxing","other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan

# Convert to numeric (coerce invalid text to NaN)
for col in ["Social2","working","other_working","relaxing","other_relaxing"]:
    activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")

# Compute interpretable scores (use min_count=1 so all-NaN stays NaN)
activity_clean["workload_score"] = activity_clean[["working","other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing","other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]

print("\nAfter computing workload_score / recovery_score / social_score:")
print(activity_clean[["student_id","timestamp","workload_score","recovery_score","social_score"]].head())

# Keep relevant columns and drop rows where ALL scores are missing
activity_clean = activity_clean[["student_id","timestamp","workload_score","recovery_score","social_score"]]
activity_clean = activity_clean.dropna(subset=["workload_score","recovery_score","social_score"], how="all").copy()

print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())
# =========================
# SLEEP DATASET
# =========================
print("\n📌 DATASET: SLEEP")
print("➡ Shows students' self-reported sleep duration in hours")

print("\nCleaning StudentLife Sleep...")

sleep_clean = sleep_raw.copy()
# Drop the export-artifact column named 'null' (irrelevant for modeling)
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

# Clean student_id
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert sleep hours to numeric, drop NaNs
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


# =========================
# FINAL SUMMARY
# =========================
print("\n" + "="*70)
print("✅ FINAL CLEANED DATASET SHAPES")
print(f"Stress   : {stress_clean.shape}")
print(f"Activity : {activity_clean.shape}")
print(f"Sleep    : {sleep_clean.shape}")
print("="*70)


---
# SECTION 4: FEATURE ENGINEERING (SIMPLIFIED)

In [ ]:
import pandas as pd
import numpy as np

print("="*70)
print("📊 CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)")
print("="*70)

print("""
Goal (easy summary):
1) Make timestamps usable (datetime)
2) Ensure numeric columns are numeric (so mean/std work)
3) Create 'week' index for each record
4) Aggregate each signal per student per week (mean / std / max / count)
5) Merge signals into ONE weekly table (this is our StudentLife multimodal dataset)
""")

# ==============================
# STEP 0: Ensure timestamp is datetime
# ==============================
for df, time_col in [
    (stress_clean, "timestamp"),
    (activity_clean, "timestamp"),
    (sleep_clean, "timestamp"),
]:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")


# ==============================

# ==============================
# STEP 1B: Per-student baseline normalization (VERY IMPORTANT)
# ==============================
# Why?
# - Students have different personal baselines (some always sleep less, some always report higher stress).
# - Burnout risk is better captured as a *deviation from personal normal* rather than absolute level.
#
# We create z-scores per student:
#   z = (value - student_mean) / student_std
# If a student_std is 0 or missing, we replace it with 1 to avoid division errors.

def add_student_zscore(df, id_col, value_col, new_col, time_col=None):
    """Causal (leakage-safe) z-score per student.

    Uses expanding mean/std in chronological order so each record is normalized
    using only that student's history up to the current timestamp (no future leakage).
    """
    if time_col is None:
        # fall back to index order if timestamp isn't provided
        time_col = None

    if time_col is not None and time_col in df.columns:
        df = df.sort_values([id_col, time_col]).copy()
    else:
        df = df.sort_values([id_col]).copy()

    def _exp_z(s: pd.Series) -> pd.Series:
        exp_mean = s.expanding(min_periods=1).mean()
        exp_std = s.expanding(min_periods=2).std().fillna(0.0)
        exp_std = exp_std.replace(0, np.nan).fillna(1.0)
        return (s - exp_mean) / exp_std

    df[new_col] = df.groupby(id_col)[value_col].transform(_exp_z)
    return df

stress_clean = add_student_zscore(stress_clean, "student_id", "stress_level", "stress_z", time_col="timestamp")
for col, zcol in [("workload_score","workload_z"),("recovery_score","recovery_z"),("social_score","social_z")]:
    if col in activity_clean.columns:
        activity_clean = add_student_zscore(activity_clean, "student_id", col, zcol, time_col="timestamp")
sleep_clean = add_student_zscore(sleep_clean, "student_id", "sleep_hours", "sleep_z")

# STEP 2: Create week index
# ==============================
# We use ISO week number; this is stable and easy for weekly aggregation.
stress_clean["week"] = stress_clean["timestamp"].dt.isocalendar().week.astype(int)
activity_clean["week"] = activity_clean["timestamp"].dt.isocalendar().week.astype(int)
sleep_clean["week"] = sleep_clean["timestamp"].dt.isocalendar().week.astype(int)

# ==============================
# STEP 3: Weekly STRESS aggregation
# ==============================
stress_weekly = (
    stress_clean.groupby(["student_id", "week"])
    .agg(
        stress_mean=("stress_level","mean"),
        stress_std=("stress_level","std"),
        stress_max=("stress_level","max"),
        stress_count=("stress_level","count"),
        stress_z_mean=("stress_z","mean"),
        stress_z_std=("stress_z","std"),
        stress_z_max=("stress_z","max"),
    )
    .reset_index()
)

# ==============================
# ==============================
# STEP 4: Weekly ACTIVITY-related aggregation (interpretable EMA scores)
# ==============================
# We aggregate 3 separate scores:
# - workload_score  (working + other_working)
# - recovery_score  (relaxing + other_relaxing)
# - social_score    (Social2)
#
# Each produces weekly mean/std/count plus z-score summaries.

agg_dict = {}
for prefix, val_col, z_col in [
    ("workload", "workload_score", "workload_z"),
    ("recovery", "recovery_score", "recovery_z"),
    ("social", "social_score", "social_z"),
]:
    if val_col in activity_clean.columns:
        agg_dict.update({
            f"{prefix}_mean": (val_col, "mean"),
            f"{prefix}_std": (val_col, "std"),
            f"{prefix}_max": (val_col, "max"),
            f"{prefix}_count": (val_col, "count"),
        })
    if z_col in activity_clean.columns:
        agg_dict.update({
            f"{prefix}_z_mean": (z_col, "mean"),
            f"{prefix}_z_std": (z_col, "std"),
            f"{prefix}_z_max": (z_col, "max"),
        })

activity_weekly = (
    activity_clean.groupby(["student_id", "week"])
    .agg(**agg_dict)
    .reset_index()
)
# ==============================
# STEP 5: Weekly SLEEP aggregation
# ==============================
sleep_weekly = (
    sleep_clean.groupby(["student_id", "week"])
    .agg(
        sleep_mean=("sleep_hours","mean"),
        sleep_std=("sleep_hours","std"),
        sleep_count=("sleep_hours","count"),
        sleep_z_mean=("sleep_z","mean"),
        sleep_z_std=("sleep_z","std"),
    )
    .reset_index()
)

# ==============================
# STEP 6: Merge into ONE multimodal weekly table
# ==============================
studentlife_features = (
    stress_weekly
    .merge(activity_weekly, on=["student_id", "week"], how="left")
    .merge(sleep_weekly, on=["student_id", "week"], how="left")
)

# This is our multimodal dataset (within StudentLife).
multimodal_df = studentlife_features.copy()

# ==============================
# STEP 6.1: Fix count NaNs + add missingness flags (important!)
# ------------------------------
# When a modality has no records for a student-week, *_count becomes NaN after merge.
# NaN should mean "0 observations" for counts. We convert NaN->0 and add explicit missing flags.
count_cols = [
    "stress_count", "sleep_count",
    "workload_count", "recovery_count", "social_count"
]
for c in count_cols:
    if c in multimodal_df.columns:
        multimodal_df[c] = multimodal_df[c].fillna(0).astype(float)

# Missing flags (1 = no observation that week for that modality)
if "stress_count" in multimodal_df.columns:
    multimodal_df["stress_missing"] = (multimodal_df["stress_count"] == 0).astype(int)
if "sleep_count" in multimodal_df.columns:
    multimodal_df["sleep_missing"] = (multimodal_df["sleep_count"] == 0).astype(int)
if "workload_count" in multimodal_df.columns:
    multimodal_df["workload_missing"] = (multimodal_df["workload_count"] == 0).astype(int)
if "recovery_count" in multimodal_df.columns:
    multimodal_df["recovery_missing"] = (multimodal_df["recovery_count"] == 0).astype(int)
if "social_count" in multimodal_df.columns:
    multimodal_df["social_missing"] = (multimodal_df["social_count"] == 0).astype(int)


# ==============================
# STEP X: Make "no-observation weeks" explicit (CRITICAL)
# If *_count == 0, then the corresponding weekly summary stats should be 0 (not imputed later).
# This preserves informative missingness (disengagement) and avoids turning missing behavior into "normal" behavior.
# ==============================
for prefix in ["stress", "sleep", "workload", "recovery", "social"]:
    ccount = f"{prefix}_count"
    if ccount in multimodal_df.columns:
        mask0 = multimodal_df[ccount] == 0
        for stat in ["mean", "std", "max"]:
            cstat = f"{prefix}_{stat}"
            if cstat in multimodal_df.columns:
                multimodal_df.loc[mask0, cstat] = 0.0
        # If any z-scored weekly columns exist, zero them too
        for zc in [f"{prefix}_z", f"{prefix}_mean_z", f"{prefix}_std_z", f"{prefix}_max_z"]:
            if zc in multimodal_df.columns:
                multimodal_df.loc[mask0, zc] = 0.0

# NOTE: Means/std/max remain NaN when missing; the sklearn imputer handles them later.



    # --- DEMO-ALIGNED FEATURES (For Production App) ---
    multimodal_df["stress_score"] = multimodal_df["stress_mean"].fillna(2.5)
    multimodal_df["workload_score"] = multimodal_df["workload_mean"].fillna(3.0)
    multimodal_df["sleep_deficit"] = (8.0 - multimodal_df["sleep_mean"]).clip(lower=0)
    multimodal_df["activity_deficit"] = (3.0 - multimodal_df["recovery_mean"]).clip(lower=0)
    social_score = (multimodal_df.get("social_mean", 2.5) - 1.0).clip(0, 3)
    multimodal_df["social_deficit"] = (3.0 - social_score).clip(lower=0)
    DEMO_ALIGNED_COLS = ["sleep_deficit", "stress_score", "workload_score", "activity_deficit", "social_deficit"]
    # --------------------------------------------------

print("Weekly multimodal StudentLife dataset created")
print("Shape:", multimodal_df.shape)
print(multimodal_df.head(3))

print("\nMissing values per column (quick check):")
print(multimodal_df.isna().sum())


In [ ]:
# STEP 7: Add temporal features (realistic + leakage-safe)
print("="*70)
print("ADDING TEMPORAL FEATURES")
print("="*70)

# Sort so rolling/diff works correctly per student
multimodal_df = multimodal_df.sort_values(["student_id", "week"]).reset_index(drop=True)

base_cols = [c for c in [
    "stress_mean","sleep_mean",
    "workload_mean","recovery_mean","social_mean",
    "stress_z_mean","sleep_z_mean",
    "workload_z_mean","recovery_z_mean","social_z_mean"] if c in multimodal_df.columns]

# 1) Simple week-to-week change (diff)
for col in base_cols:
    multimodal_df[f"{col}_diff1"] = multimodal_df.groupby("student_id")[col].diff(1)
    multimodal_df[f"{col}_diff2"] = multimodal_df.groupby("student_id")[col].diff(2)

# 2) Rolling stability features (mean/std over last 4 weeks)
window = 4
for col in base_cols:
    multimodal_df[f"{col}_roll{window}_mean"] = (
        multimodal_df.groupby("student_id")[col]
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True))

    multimodal_df[f"{col}_roll{window}_std"] = (
        multimodal_df.groupby("student_id")[col]
        .rolling(window=window, min_periods=2)
        .std()
        .reset_index(level=0, drop=True))

# 3) Missingness / data-availability features (important for sensor/self-report data)
# If a student stops reporting, that itself can carry information.
for c in ["stress_count","sleep_count","workload_count","recovery_count","social_count"]:
    if c in multimodal_df.columns:
        multimodal_df[f"{c}_is_low"] = (multimodal_df[c] <= 1).astype(int)

print("✅ Temporal features added")
print("Final shape:", multimodal_df.shape)
print("Example columns:", [c for c in multimodal_df.columns if "diff" in c or "roll_" in c][:12])


---
# SECTION 5: CREATE TARGET VARIABLE

In [ ]:
# Define proxy burnout risk label (multi-signal, 2-week-ahead)
print("="*70)
print("CREATING TARGET LABEL (PROXY - RELAXED)")
print("="*70)

print("""
Reality check (important):
- We use a relaxed target definition to ensure the model sees enough positive examples.
- Target: ~25% positives.
""")

# Two-week-ahead outcomes
multimodal_df["future_stress_2w"] = multimodal_df.groupby("student_id")["stress_mean"].shift(-2)
multimodal_df["future_sleep_2w"]  = multimodal_df.groupby("student_id")["sleep_mean"].shift(-2)

# Keep only rows where we can define the 2-week-ahead outcome
multimodal_df = multimodal_df.dropna(subset=["future_stress_2w", "future_sleep_2w"]).reset_index(drop=True)

# Relaxed quantiles for ~25% positive rate
stress_q_candidates = [0.55, 0.60, 0.65, 0.70]
sleep_q_candidates  = [0.45, 0.40, 0.35, 0.30]

target_rate = 0.25
min_rate, max_rate = 0.18, 0.40

best = None
for sq in stress_q_candidates:
    stress_th = multimodal_df["future_stress_2w"].quantile(sq)
    for lq in sleep_q_candidates:
        sleep_th = multimodal_df["future_sleep_2w"].quantile(lq)
        # Use relaxed AND logic (both stress and sleep issues)
        y = ((multimodal_df["future_stress_2w"] >= stress_th) &
             (multimodal_df["future_sleep_2w"]  <= sleep_th)).astype(int)
        pos_rate = float(y.mean())
        pos_count = int(y.sum())
        in_range = (min_rate <= pos_rate <= max_rate)
        score = abs(pos_rate - target_rate) + (0 if in_range else 0.5)
        if best is None or score < best[0]:
            best = (score, sq, lq, float(stress_th), float(sleep_th), pos_rate, pos_count)

_, STRESS_Q, SLEEP_Q, STRESS_TH, SLEEP_TH, POS_RATE, POS_COUNT = best

multimodal_df["burnout_risk"] = (
    (multimodal_df["future_stress_2w"] >= STRESS_TH) &
    (multimodal_df["future_sleep_2w"]  <= SLEEP_TH)
).astype(int)

print(f"Positive rate: {POS_RATE:.3f} | Positives: {POS_COUNT} / {len(multimodal_df)}")
print(multimodal_df["burnout_risk"].value_counts())


---
# SECTION 6: PREPARE DATA FOR MODELING

In [ ]:
# Prepare feature sets (baseline vs multimodal)
print("="*70)
print("🧩 PREPARING FEATURE SETS")
print("="*70)

y = multimodal_df["burnout_risk"].astype(int)

# ------------------------------
# Baseline: stress-only (what if we only use stress?)
# ------------------------------

# ------------------------------
# Three feature sets (Ablation Study)
# A) Stress-only
# B) Stress + Sleep
# C) Stress + Sleep + Activity (Full Multimodal)
# ------------------------------

baseline_cols = [
    # raw stress stats
    "stress_mean","stress_std","stress_max","stress_count",
    # normalized stress stats (deviation from personal baseline)
    "stress_z_mean","stress_z_std","stress_z_max",
    # temporal stress features (raw + z)
    "stress_mean_diff1","stress_mean_diff2",
    "stress_z_mean_diff1","stress_z_mean_diff2",
    "stress_mean_roll4_mean","stress_mean_roll4_std",
    "stress_z_mean_roll4_mean","stress_z_mean_roll4_std",
    # missingness flags
    "stress_missing"

]

stress_sleep_cols = baseline_cols + [
    "sleep_mean","sleep_std","sleep_count",
    "sleep_z_mean","sleep_z_std",
    "sleep_mean_diff1","sleep_mean_diff2",
    "sleep_z_mean_diff1","sleep_z_mean_diff2",
    "sleep_mean_roll4_mean","sleep_mean_roll4_std",
    "sleep_z_mean_roll4_mean","sleep_z_mean_roll4_std",
]

multimodal_cols = stress_sleep_cols + [
    # Workload (working-related)
    "workload_mean","workload_std","workload_max","workload_count",
    "workload_z_mean","workload_z_std","workload_z_max",
    "workload_mean_diff1","workload_mean_diff2",
    "workload_z_mean_diff1","workload_z_mean_diff2",
    "workload_mean_roll4_mean","workload_mean_roll4_std",
    "workload_z_mean_roll4_mean","workload_z_mean_roll4_std",

    # Recovery (relaxing-related)
    "recovery_mean","recovery_std","recovery_max","recovery_count",
    "recovery_z_mean","recovery_z_std","recovery_z_max",
    "recovery_mean_diff1","recovery_mean_diff2",
    "recovery_z_mean_diff1","recovery_z_mean_diff2",
    "recovery_mean_roll4_mean","recovery_mean_roll4_std",
    "recovery_z_mean_roll4_mean","recovery_z_mean_roll4_std",

    # Social interaction (Social2)
    "social_mean","social_std","social_max","social_count",
    "social_z_mean","social_z_std","social_z_max",
    "social_mean_diff1","social_mean_diff2",
    "social_z_mean_diff1","social_z_mean_diff2",
    "social_mean_roll4_mean","social_mean_roll4_std",
    "social_z_mean_roll4_mean","social_z_mean_roll4_std",
]

# Add missingness flags (if present)
multimodal_cols += ["workload_missing","recovery_missing","social_missing"]

# Keep only columns that exist (not all datasets have all stats)
baseline_cols = [c for c in baseline_cols if c in multimodal_df.columns]
stress_sleep_cols = [c for c in stress_sleep_cols if c in multimodal_df.columns]
multimodal_cols = [c for c in multimodal_cols if c in multimodal_df.columns]

X_baseline = multimodal_df[baseline_cols].copy()
X_stress_sleep = multimodal_df[stress_sleep_cols].copy()
X_multimodal = multimodal_df[multimodal_cols].copy()

groups = multimodal_df["student_id"]
weeks = multimodal_df["week"]

print("✅ Feature sets ready:")
print("  A) Stress-only:", X_baseline.shape)
print("  B) Stress+Sleep:", X_stress_sleep.shape)
print("  C) Full Multimodal:", X_multimodal.shape)


---
# SECTION 7: GRIDSEARCH FOR BEST MODELS

In [ ]:
# Updated model parameters with strict regularization to prevent overfitting
models_and_params = {
    "LogReg": {
        "model": LogisticRegression(max_iter=2000, class_weight=None),
        "params": {
            "model__C": [0.01, 0.1, 1.0]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5],
            "model__min_samples_leaf": [5, 10]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.01, 0.05],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [5, 10]
        }
    },
    "SVM": {
        "model": SVC(probability=True),
        "params": {
            "model__C": [0.1, 0.5, 1.0],
            "model__kernel": ["linear", "rbf"]
        }
    }
}


In [ ]:
def run_gridsearch(X, y, groups, model_dict, dataset_name, weeks=None):
    """
    Methodologically clean GridSearch with 3-way Grouped Splitting and Calibration.
    """
    print(f"\n{'='*70}")
    print(f"GRIDSEARCH: {dataset_name}")
    print(f"{'='*70}")

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    temp_idx, test_idx = next(gss1.split(X, y, groups=groups))
    X_temp, X_test = X.iloc[temp_idx], X.iloc[test_idx]
    y_temp, y_test = y.iloc[temp_idx], y.iloc[test_idx]
    groups_temp, groups_test = groups.iloc[temp_idx], groups.iloc[test_idx]

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, val_idx = next(gss2.split(X_temp, y_temp, groups=groups_temp))
    X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
    y_train, y_val = y_temp.iloc[train_idx], y_temp.iloc[val_idx]
    groups_train = groups_temp.iloc[train_idx]

    print(f"  [INFO] Splits: Train={len(y_train)}, Val={len(y_val)}, Test={len(y_test)}")

    results = {}
    best_model = None
    best_score = -1.0
    best_name = None
    cv = GroupKFold(n_splits=3)

    for name, config in model_dict.items():
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy='mean')),
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=42)),
            ("model", config['model']),
        ])
        grid = GridSearchCV(pipeline, config['params'], cv=cv.split(X_train, y_train, groups=groups_train), scoring='roc_auc', n_jobs=-1)
        grid.fit(X_train, y_train)

        y_val_proba = grid.predict_proba(X_val)[:, 1]
        val_roc = roc_auc_score(y_val, y_val_proba) if y_val.sum() > 0 else 0.5
        print(f"    {name} -> Val ROC: {val_roc:.4f}")
        if val_roc > best_score:
            best_score = val_roc
            best_model = grid.best_estimator_
            best_name = name

    if best_model:
        calib = CalibratedClassifierCV(best_model, method='sigmoid', cv='prefit')
        calib.fit(X_val, y_val)
        best_model = calib

    y_test_proba = best_model.predict_proba(X_test)[:, 1]
    final_roc = roc_auc_score(y_test, y_test_proba) if y_test.sum() > 0 else 0.5
    print(f"  --- FINAL TEST SCORE ({best_name}): {final_roc:.4f} ---")

    return {best_name: {'model': best_model, 'roc_auc': final_roc, 'X_test': X_test, 'y_test': y_test}}, best_model, best_name


In [ ]:
# Run GridSearch for Baseline (stress-only)
print("="*70)
print("1️⃣ GRIDSEARCH: BASELINE (STRESS-ONLY)")
print("="*70)

results_studentlife, best_studentlife, best_name_studentlife = run_gridsearch(
    X_baseline, y, groups, models_and_params, "Baseline", weeks=weeks
)


In [ ]:

# Run GridSearch for Stress + Sleep (ablation middle)
print("\n" + "="*70)
print("2️⃣ GRIDSEARCH: STRESS + SLEEP (ABLATION)")
print("="*70)

results_stress_sleep, best_stress_sleep, best_name_stress_sleep = run_gridsearch(
    X_stress_sleep, y, groups, models_and_params, "Stress+Sleep", weeks=weeks
)


In [ ]:
# Run GridSearch for Multimodal (stress + sleep + activity)
print("\n" + "="*70)
print("3️⃣ GRIDSEARCH: MULTIMODAL (STRESS + SLEEP + ACTIVITY)")
print("="*70)

results_multimodal, best_multimodal, best_name_multimodal = run_gridsearch(
    X_multimodal, y, groups, models_and_params, "Multimodal", weeks=weeks
)

from scipy.optimize import minimize
def train_demo_constrained(X, y):
    imp = SimpleImputer(strategy='mean').fit(X)
    scl = StandardScaler().fit(imp.transform(X))
    X_const = np.column_stack([np.ones(X.shape[0]), scl.transform(imp.transform(X))])
    def obj(w, X, y):
        p = 1 / (1 + np.exp(-np.dot(X, w)))
        return -np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15)) + 0.01 * np.sum(w[1:]**2)
    bounds = [(None, None), (0.1, None), (0.1, None), (0.1, None), (0.1, None), (0.3, None)]
    res = minimize(obj, np.zeros(X_const.shape[1]), args=(X_const, y), bounds=bounds, method='L-BFGS-B')
    return res.x, imp, scl

print("\n" + "="*70)
print("4️⃣ FINAL DEMO-ALIGNED MODEL (CONSTRAINED)")
print("="*70)
X_demo = multimodal_df[DEMO_ALIGNED_COLS].copy()
w_demo, imp_demo, scl_demo = train_demo_constrained(X_demo, y)
print("\nDemo Coefficients:")
for name, val in zip(DEMO_ALIGNED_COLS, w_demo[1:]): print(f"  {name:20}: {val:+.4f}")


---
# SECTION 8: RESULTS COMPARISON

In [ ]:
# Create comparison table (Baseline vs Multimodal)
print("="*70)
print("📈 MODEL COMPARISON")
print("="*70)

comparison_rows = []

def add_rows(feature_set_name, results_dict):
    for model_name, info in results_dict.items():
        comparison_rows.append({
            "FeatureSet": feature_set_name,
            "Model": model_name,
            "BestCVScore(ROC-AUC)": info.get("roc_auc", np.nan),
            "TestPR_AUC": info.get("pr_auc", np.nan),
            "TestF1": info.get("f1", np.nan),
            "TestAccuracy": info.get("accuracy", np.nan),
        })

add_rows("Baseline (Stress-only)", results_studentlife)
add_rows("Multimodal (Stress+Sleep+Activity)", results_multimodal)

comparison_df = pd.DataFrame(comparison_rows).sort_values(["FeatureSet","BestCVScore(ROC-AUC)"], ascending=[True, False])
print(comparison_df)


In [ ]:
print("="*70)
print("⏳ EARLY-WARNING EVALUATION (LEAD-TIME)")
print("="*70)

from sklearn.metrics import brier_score_loss, calibration_curve

# 1. Extract best multimodal bundle
bundle = results_multimodal[best_name_multimodal]
best_pipe = bundle['model']
X_p = bundle['X_test']
y_p = bundle['y_test']
g_p = bundle['groups_test']
w_p = bundle['weeks_test']

if w_p is None:
    print("WARNING: Lead-time metrics skipped (no weeks metadata available)")
else:
    # Predict probabilities
    probs = best_pipe.predict_proba(X_p)[:, 1]
    preds = (probs >= 0.5).astype(int)

    # Build evaluation frame
    df_ev = pd.DataFrame({
        'sid': g_p, 'wk': w_p, 'y_true': y_p, 'y_pred': preds
    }).sort_values(['sid', 'wk']).reset_index(drop=True)

    # Calculate lead-time performance
    df_ev['prev_y'] = df_ev.groupby('sid')['y_true'].shift(1).fillna(0)
    df_ev['onset'] = ((df_ev['y_true']==1) & (df_ev['prev_y']==0)).astype(int)

    onsets = df_ev[df_ev['onset']==1]
    if len(onsets) == 0:
        print("No onset events in test set.")
    else:
        caught = 0
        for i, row in onsets.iterrows():
            # Check if predicted positive at t-1, t-2, or t-3
            sid = row['sid']
            # Getting sequential history for this student up to this onset
            idx = df_ev[(df_ev['sid']==sid)].index.tolist()
            pos = idx.index(i)
            history = df_ev.iloc[max(0, i-3):i]
            if (history['y_pred']==1).any():
                caught += 1
        print(f"Early Detection Rate (within 3 weeks): {caught/len(onsets):.3f} ({caught}/{len(onsets)})")

# 2. Brier score for calibration quality
pp = best_pipe.predict_proba(X_p)[:, 1]
print(f"Brier Score: {brier_score_loss(y_p, pp):.4f} (lower is better)")



---
# SECTION 9: SHAP EXPLAINABILITY

In [ ]:
# SHAP Explainability (ONLY for the final multimodal model)
print("="*70)
print("🔍 SHAP EXPLAINABILITY: MULTIMODAL MODEL ONLY")
print("="*70)

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pull splits + best pipeline from stored results
best_name = best_name_multimodal
bundle = results_multimodal[best_name]
best_pipe = bundle["model"]

X_test_mm = bundle["X_test"]
y_test_mm = bundle["y_test"]
X_train_mm = bundle.get("X_train", None)  # may or may not be stored

# ---- Extract preprocessing + model (the model was trained on *transformed* features)
imputer = best_pipe.named_steps["imputer"]
scaler  = best_pipe.named_steps["scaler"]
model   = best_pipe.named_steps["model"]

# Transform data exactly the same way as training
X_test_trans = scaler.transform(imputer.transform(X_test_mm))

# Background data: ideally from TRAIN (more correct than test). Fallback to test.
if X_train_mm is not None:
    X_bg_base = X_train_mm
else:
    X_bg_base = X_test_mm

X_bg_trans = scaler.transform(imputer.transform(X_bg_base))

# Use a small random background sample for speed + stability
rng = np.random.default_rng(42)
bg_size = min(200, X_bg_trans.shape[0])
bg_idx = rng.choice(X_bg_trans.shape[0], size=bg_size, replace=False)
background = X_bg_trans[bg_idx]

feature_names = list(X_test_mm.columns)

print("Best multimodal model:", best_name)
print("X_test shape:", X_test_mm.shape)
print("Transformed X_test shape:", X_test_trans.shape)
print("Background shape:", background.shape)

# ------------------------------
# Choose an explainer (model-dependent)
# ------------------------------
is_tree = best_name in ["RandomForest", "XGBoost", "GradientBoosting"]
is_linear = best_name == "LogReg"

# Important note (why your error happened):
# TreeExplainer sometimes fails the "additivity check" for some sklearn models (notably GradientBoosting)
# due to numerical / approximation differences. This does NOT mean your shapes are wrong.
# We safely disable that check for plotting.
if is_tree:
    explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")
    shap_values = explainer.shap_values(X_test_trans, check_additivity=False)
elif is_linear:
    explainer = shap.LinearExplainer(model, background)
    shap_values = explainer.shap_values(X_test_trans)
else:
    # Generic fallback (slow) for models like SVM: KernelExplainer on probability of class 1
    def predict_proba_class1(x):
        return model.predict_proba(x)[:, 1]
    # Cap to keep runtime reasonable
    X_explain = X_test_trans[:200]
    explainer = shap.KernelExplainer(predict_proba_class1, background)
    shap_values = explainer.shap_values(X_explain)
    X_test_trans = X_explain  # keep shapes aligned for plotting

# ------------------------------
# Global summary plot
# ------------------------------
print("\n📌 Global feature importance (SHAP summary)...")
plt.figure()
# TreeExplainer may return list for some models; for binary classification, use class 1 if present.
if isinstance(shap_values, list):
    if len(shap_values) > 1:
        shap_vals_to_plot = shap_values[1]
    else:
        shap_vals_to_plot = shap_values[0]
else:
    shap_vals_to_plot = shap_values

shap.summary_plot(shap_vals_to_plot, X_test_trans, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()

# ------------------------------
# One local explanation example
# ------------------------------
print("\n📌 One local explanation example (first test sample)...")
i = 0
try:
    shap.plots.waterfall(shap.Explanation(
        values=shap_vals_to_plot[i],
        base_values=getattr(explainer, "expected_value", 0) if not isinstance(getattr(explainer, "expected_value", 0), (list, np.ndarray)) else (explainer.expected_value[1] if len(explainer.expected_value) > 1 else explainer.expected_value[0]),
        data=X_test_trans[i],
        feature_names=feature_names
    ))
except Exception as e:
    print("Waterfall plot not available for this explainer in this environment:", e)


In [ ]:
print("\n--- Baseline Model y_test Class Distribution ---")
y_test_baseline = results_studentlife[best_name_studentlife]['y_test'] if best_name_studentlife else results_studentlife['LogReg']['y_test'] # Default to LogReg if best_name is None
print(y_test_baseline.value_counts())

print("\n--- Multimodal Model y_test Class Distribution ---")
y_test_multimodal = results_multimodal[best_name_multimodal]['y_test'] if best_name_multimodal else results_multimodal['LogReg']['y_test'] # Default to LogReg if best_name is None
print(y_test_multimodal.value_counts())

print("\nAfter re-running, the distribution should show positive samples if the resampling was successful.")

---
# SECTION 10: SAVE RESULTS

In [ ]:
'''
# Save all results
print("="*70)
print("💾 SAVING RESULTS")
print("="*70)

import pickle

# Save comparison table
comparison_df.to_csv(f"{OUTPUT_PATH}/model_comparison.csv", index=False)
print("✅ Saved: model_comparison.csv")

# Save dataset
multimodal_df.to_csv(f"{OUTPUT_PATH}/multimodal_features.csv", index=False)
print("✅ Saved: multimodal_features.csv")

# Save best model info
with open(f"{OUTPUT_PATH}/best_models_info.txt", 'w') as f:
    f.write("BEST MODELS FROM GRIDSEARCH\n")
    f.write("="*50 + "\n\n")
    f.write(f"StudentLife Best Model: {best_name_studentlife}\n")
    f.write(f"  Parameters: {results_studentlife[best_name_studentlife]['best_params']}\n")
    f.write(f"  ROC AUC: {results_studentlife[best_name_studentlife]['roc_auc']:.4f}\n\n")

    if best_name_oulad:
        f.write(f"OULAD Best Model: {best_name_oulad}\n")
        f.write(f"  Parameters: {results_oulad[best_name_oulad]['best_params']}\n")
        f.write(f"  ROC AUC: {results_oulad[best_name_oulad]['roc_auc']:.4f}\n\n")

    f.write(f"Multimodal Best Model: {best_name_multimodal}\n")
    f.write(f"  Parameters: {results_multimodal[best_name_multimodal]['best_params']}\n")
    f.write(f"  ROC AUC: {results_multimodal[best_name_multimodal]['roc_auc']:.4f}\n")

print("✅ Saved: best_models_info.txt")

# Save models
with open(f"{OUTPUT_PATH}/best_studentlife_model.pkl", 'wb') as f:
    pickle.dump(best_studentlife, f)
print("✅ Saved: best_studentlife_model.pkl")

if best_oulad:
    with open(f"{OUTPUT_PATH}/best_oulad_model.pkl", 'wb') as f:
        pickle.dump(best_oulad, f)
    print("✅ Saved: best_oulad_model.pkl")

with open(f"{OUTPUT_PATH}/best_multimodal_model.pkl", 'wb') as f:
    pickle.dump(best_multimodal, f)
print("✅ Saved: best_multimodal_model.pkl")

print("\n" + "="*70)
print("✅ ALL RESULTS SAVED TO: " + OUTPUT_PATH)
print("="*70)
'''

---
# SECTION 11: FINAL SUMMARY

In [ ]:
# Print final summary
print("="*70)
print("🎉 ANALYSIS COMPLETE - SUMMARY")
print("="*70)

print("\n📊 Dataset Information:")
print(f"  Total samples (student-week rows): {len(multimodal_df)}")
print(f"  Burnout-risk cases: {multimodal_df['burnout_risk'].sum()} ({multimodal_df['burnout_risk'].mean()*100:.1f}%) -- Note: This is overall dataset, not just test set.")
print(f"  Unique students: {multimodal_df['student_id'].nunique()}")

print("\n🎯 Feature Breakdown:")
print(f"  Baseline (stress-only) features: {X_baseline.shape[1]}")
print(f"  Multimodal (stress+sleep+activity) features: {X_multimodal.shape[1]}")

print("\n🏆 Best Models:")
sl_auc = results_studentlife[best_name_studentlife]['roc_auc']
mm_auc = results_multimodal[best_name_multimodal]['roc_auc']

sl_auc_str = f"{sl_auc:.4f}" if not np.isnan(sl_auc) else "N/A"
mm_auc_str = f"{mm_auc:.4f}" if not np.isnan(mm_auc) else "N/A"

print(f"  Baseline best:   {best_name_studentlife} (ROC-AUC: {sl_auc_str})")
print(f"  Multimodal best: {best_name_multimodal} (ROC-AUC: {mm_auc_str})")

print("\n✅ Explainability:")
print("  SHAP was run ONLY for the best multimodal model (as intended).")
print(f"\n\u2705 Demo Alignment:")
print(f"  Demo specific features: {len(DEMO_ALIGNED_COLS)}")
print(f"  Social Deficit Factor:  {w_demo[5]:.4f} (Amplified for Demo)")


# Task
Reinforce the numeric type conversion for `stress_level` in the `hrcfseqYMy1C` cell by explicitly casting the column to `float` after converting it to numeric and dropping NaNs, then re-run the `hrcfseqYMy1C` and `3lNsZ2kgMy1E` cells.

## Reinforce numeric type conversion for stress_level

### Subtask:
Reinforce the numeric type conversion for `stress_level` in the `hrcfseqYMy1C` cell by explicitly casting the column to `float` after converting it to numeric and dropping NaNs, then re-run the `hrcfseqYMy1C` and `3lNsZ2kgMy1E` cells.


## Summary:

The provided information describes a subtask to reinforce the numeric type conversion for the `stress_level` column. No analysis has been performed yet, so there are no findings or insights to report at this stage.
